# NB 1.1 &mdash; Presa de contacte amb les dades

**MP 5134 &mdash; Disseny i avaluació de models basats en aprenentatge automàtic**
UT1: Entorn de treball i primer model de principi a fi

---

### Què farem avui

Res d'entrenar models. Avui només mirem dades. Sembla poc, però és la meitat de
la feina real d'un projecte d'aprenentatge automàtic: si no entens què hi ha
dins del fitxer, qualsevol model que entrenis serà una loteria.

En acabar aquest notebook has de saber respondre:

1. Quantes mostres i quantes característiques té el nostre conjunt de dades?
2. Què volem predir exactament?
3. Quines columnes tenen valors absents?
4. Per què el problema de la pluja no està equilibrat?

## 1. De què va tot això

L'aprenentatge automàtic no és més que això: **tenim exemples del passat i volem
fer prediccions sobre casos nous**.

La diferència amb la programació que ja coneixeu és on posem les regles.

| Programació clàssica | Aprenentatge automàtic |
|---|---|
| Tu escrius les regles | L'algorisme les dedueix dels exemples |
| Dades + regles &rarr; resposta | Dades + respostes &rarr; regles |

Si volguéssim predir la temperatura de demà amb programació clàssica, hauríem
d'escriure nosaltres les condicions: *si la humitat és alta i la pressió baixa,
llavors...*. Amb aprenentatge automàtic li donem milers de dies passats i deixem
que l'algorisme trobi el patró.

Això té una conseqüència important: **el model només serà tan bo com les dades
que li donem**. D'aquí que avui dediquem tota una sessió a mirar-les.

## 2. El nostre conjunt de dades

Farem servir dades meteorològiques diàries reals de l'estació **B278**,
l'aeroport de Palma, publicades per l'AEMET.

Són mesures, no estimacions ni valors calculats. Això importa més del que sembla,
i ho veurem al llarg del curs.

Cada fila és **un dia**. Les columnes principals:

| Columna | Què és |
|---|---|
| `fecha` | data del dia |
| `mes`, `dia_any` | variables de calendari |
| `tmed`, `tmin`, `tmax` | temperatura mitjana, mínima i màxima (&deg;C) |
| `prec` | precipitació (mm) |
| `velmedia`, `racha` | vent mitjà i ratxa màxima (m/s) |
| `sol` | hores de sol |
| `presMax`, `presMin` | pressió màxima i mínima (hPa) |
| `tmax_dema` | **temperatura màxima del dia següent** |
| `prec_dema` | precipitació del dia següent |
| `plou_dema` | 1 si demà plou (&ge; 1 mm), 0 si no |

In [2]:
import pandas as pd
import matplotlib.pyplot as plt

# Dades diàries de l'estació B278 (aeroport de Palma), publicades per l'AEMET
URL_DADES = "https://raw.githubusercontent.com/pprohenspolitecnicllevant/disseny-avaluacio-models-ml/refs/heads/main/UT01-Entorn_de_treball_primer_model/aemet/meteo_palma.csv"

df = pd.read_csv(URL_DADES, parse_dates=["fecha"])
print("Dades carregades correctament.")

Matplotlib is building the font cache; this may take a moment.


Dades carregades correctament.


Fixa't en dues coses d'aquesta cel·la.

`pd.read_csv` accepta directament una URL: no cal descarregar res a mà. Això fa
que el notebook funcioni igual a qualsevol ordinador amb connexió.

`parse_dates=["fecha"]` li diu a Pandas que aquella columna no és text, sinó una
data. Sense això, `"2015-03-14"` seria una cadena de caràcters i no podríem
ordenar ni filtrar per períodes.

### 2.1 Quantes dades tenim

El primer que es mira sempre d'un conjunt de dades és la seva forma.

In [3]:
files, columnes = df.shape
print(f"Files:    {files}")
print(f"Columnes: {columnes}")

Files:    4017
Columnes: 16


Aquí apareix el **vocabulari fonamental** del mòdul. Val la pena fixar-lo ara,
perquè el farem servir cada dia fins al maig:

- **Mostra** o **instància**: una fila. En el nostre cas, un dia concret.
- **Característica** (*feature*): una columna que fem servir com a entrada.
- **Variable objectiu** (*target*): la columna que volem predir.

Una forma útil de recordar-ho: les mostres són *els exemples*, les
característiques són *el que sabem de cada exemple*, i la variable objectiu és
*el que volem endevinar*.

Compte amb una confusió molt freqüent: **no totes les columnes són
característiques**. `tmax_dema` és la variable objectiu i no entrarà mai com a
entrada del model.

In [4]:
df.head()

,fecha,any,mes,dia_any,tmed,tmin,tmax,prec,velmedia,racha,sol,presMax,presMin,tmax_dema,prec_dema,plou_dema
0,2015-01-01,2015,1,1,7.7,0.9,14.5,0.0,1.1,6.1,7.3,1035.9,1029.4,16.4,0.0,0
1,2015-01-02,2015,1,2,7.8,-0.8,16.4,0.0,1.1,4.7,7.9,1037.9,1034.6,16.7,0.0,0
2,2015-01-03,2015,1,3,8.2,-0.3,16.7,0.0,4.2,10.8,7.9,1036.5,1031.8,18.4,0.0,0
3,2015-01-04,2015,1,4,11.4,4.3,18.4,0.0,1.9,6.1,8.2,1031.8,1029.1,16.7,0.0,0
4,2015-01-05,2015,1,5,9.6,2.5,16.7,0.0,1.7,5.0,7.9,1031.0,1027.0,14.4,0.0,0


`head()` mostra les cinc primeres files. És el gest més repetit de tot el curs:
abans de fer res amb unes dades, mira-les.

In [ ]:
df.info()

`info()` dóna tres coses de cop:

- el **tipus** de cada columna (`float64`, `int64`, `datetime64`),
- quants valors **no nuls** té cadascuna,
- la memòria que ocupa.

Si una columna que hauria de ser numèrica apareix com a `object`, vol dir que
Pandas hi ha trobat text i no ha pogut convertir-la. És un dels errors més
habituals en carregar dades espanyoles, perquè sovint fan servir la coma com a
separador decimal.

In [ ]:
df.describe()

### 2.2 Llegir un `describe()`

Aquesta taula sembla àrida però diu moltíssim. Mira-la amb aquestes preguntes:

**Els mínims i màxims tenen sentit?** Una `tmax` de 45 &deg;C a Palma seria
sospitosa. Una de &minus;5 &deg;C, directament un error.

**La mitjana i la mediana (50%) s'assemblen?** Si són molt diferents, la
distribució està esbiaixada. Mira `prec`: la mitjana serà molt superior a la
mediana, perquè la majoria de dies no plou gens i uns pocs dies plou moltíssim.

**Els quartils estan on t'esperes?** El 25% i el 75% et diuen com es reparteixen
els valors sense necessitat de dibuixar res.

## 3. Valors absents

Cap conjunt de dades real està complet. Sensors que fallen, dies que no es va
registrar, columnes que no es van començar a mesurar fins més tard.

In [ ]:
absents = df.isna().sum()
print(absents[absents > 0].sort_values(ascending=False))
print()
print(f"Files sense cap valor absent: {df.dropna().shape[0]} de {df.shape[0]}")

Això és un problema pràctic immediat: **la majoria d'algorismes de scikit-learn
no accepten valors absents**. Si li passes una taula amb buits, peta.

Hi ha tres sortides possibles, i cadascuna té un cost:

1. **Eliminar les files** amb buits. Simple, però perds mostres.
2. **Eliminar la columna** sencera. Perds informació potencialment útil.
3. **Imputar**: omplir els buits amb la mitjana, la mediana o un valor estimat.

Avui farem servir l'opció 1 perquè és la més directa. A la **UT3** hi tornarem
amb calma, perquè la decisió no és innocent: imputar malament pot fer que el
model aprengui coses que no són certes.

## 4. Què volem predir exactament

Aquí hi ha una decisió de disseny important, i vull que la vegis des del primer
dia.

Podríem intentar predir la `tmax` d'avui a partir de la `tmed` i la `tmin`
d'avui. Però seria un exercici buit: aquestes tres variables estan lligades
gairebé per definició, i qualsevol model encertaria pràcticament sempre. Un
resultat perfecte que no serveix per a res.

Per això les nostres variables objectiu són **del dia següent**.

In [ ]:
df[["fecha", "tmax", "tmax_dema", "prec", "prec_dema", "plou_dema"]].head(8)

Mira les columnes `tmax` i `tmax_dema` d'una fila a l'altra: el valor de
`tmax_dema` d'un dia és la `tmax` del dia següent, desplaçada una posició.

Això converteix l'exercici en una **predicció de veritat**: fem servir només el
que sabem avui per dir alguna cosa sobre demà. Hi ha senyal, però no és perfecta.
Que és exactament el que necessitem per poder comparar algorismes.

### 4.1 Dos problemes sobre les mateixes dades

Amb aquest fitxer podem plantejar els dos grans tipus de problema supervisat:

- **Regressió**: predir `tmax_dema`, un número continu.
- **Classificació**: predir `plou_dema`, una etiqueta (0 o 1).

La diferència no és tècnica sinó conceptual: en regressió la resposta pot ser
qualsevol valor d'un rang; en classificació la resposta és una categoria d'un
conjunt tancat.

In [ ]:
proporcio = df["plou_dema"].value_counts(normalize=True).sort_index()
print(proporcio)
print()
print(f"Dies de pluja: {100 * proporcio.get(1, 0):.1f} %")

### 4.2 Un detall que serà clau a la UT4

Fixa't en aquesta proporció. Els dies de pluja són una minoria clara.

Això es diu **desbalanç de classes** i té una conseqüència que sorprèn molt: un
model que digui sempre *"demà no plou"*, sense mirar cap dada, encertaria la gran
majoria de vegades.

Guarda aquest número. A la **UT4** el farem servir per demostrar per què el
percentatge d'encerts és una mètrica que enganya.

## 5. Primeres gràfiques

Les taules diuen molt, però els ulls detecten coses que cap `describe()` et
donarà.

In [ ]:
plt.figure(figsize=(9, 4))
plt.hist(df["tmax_dema"].dropna(), bins=40, edgecolor="white")
plt.xlabel("Temperatura màxima de demà (°C)")
plt.ylabel("Nombre de dies")
plt.title("Distribució de la variable objectiu")
plt.show()

Un **histograma** reparteix els valors en intervals i compta quants n'hi ha a
cadascun. Serveix per veure la forma de la distribució.

Aquesta en concret no té forma de campana: té dos cims. Per què? Pensa en el
clima de Mallorca abans de continuar.

In [ ]:
un_any = df[df["any"] == df["any"].min()]

plt.figure(figsize=(11, 4))
plt.plot(un_any["fecha"], un_any["tmax"], linewidth=0.9)
plt.xlabel("Data")
plt.ylabel("Temperatura màxima (°C)")
plt.title(f"Temperatura màxima diària durant {df['any'].min()}")
plt.show()

Aquí està la resposta a la pregunta anterior: **l'estacionalitat**. Els dos cims
de l'histograma són l'hivern i l'estiu.

Això ens diu una cosa pràctica: el mes de l'any hauria de ser una característica
útil per al model. Ho comprovarem.

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(df["tmax"], df["tmax_dema"], s=3, alpha=0.25)
plt.xlabel("Temperatura màxima d'avui (°C)")
plt.ylabel("Temperatura màxima de demà (°C)")
plt.title("Hi ha relació entre avui i demà?")
plt.show()

Un **mapa de dispersió** enfronta dues variables. Cada punt és un dia.

El núvol s'estira en diagonal: quan avui fa calor, demà també sol fer-ne. És una
relació forta però no perfecta, i aquesta imprecisió és precisament l'espai on
els diferents algorismes es diferenciaran.

Aquesta idea, la de mesurar quant es relacionen dues variables, té nom propi i és
tot el contingut de la **UT3**: la correlació.

## 6. Exercicis

**1.** Quants anys diferents hi ha al conjunt? Quantes files té cada any? Fes
servir `value_counts()` sobre la columna `any`.

**2.** Quin va ser el dia més calorós de tota la sèrie? I el més plujós? Pista:
`idxmax()` et dóna la posició del màxim, i `df.loc[...]` et dóna aquella fila.

**3.** Dibuixa l'histograma de `prec`. Què li passa? Per què no s'assembla gens
al de la temperatura?

**4.** Calcula la temperatura màxima mitjana de cada mes i dibuixa-la amb un
gràfic de barres. Pista: `df.groupby("mes")["tmax"].mean()`.

**5.** Pensa i escriu la resposta: si volguéssim predir si demà plou, quines tres
columnes creus que serien més útils? Ho comprovarem d'aquí unes setmanes.